# 04 — Contrôle visuel du masque du cerveau HD-BET

Étape 3.3 de la [ROADMAP](../ROADMAP.md). HD-BET (modèle pré-entraîné, inférence seule) a produit, pour la T1 3D à 1 mm, un masque binaire du cerveau. Ce notebook montre ce masque sur de nombreuses coupes pour juger sa qualité **à l'œil** :

| Section | Contenu |
|---|---|
| 1 | Chiffres de contrôle (volume, composantes, cohérence avec le masque de tête N4) |
| 2 | Coupes axiales du bas vers le haut, contour du masque sur la T1 |
| 3 | Coupes coronales et sagittales |
| 4 | Zooms sur les zones difficiles (base du crâne, lobes temporaux, vertex) |
| 5 | Le même masque transporté sur la grille FLAIR, sur la FLAIR |

Changer de sujet : modifier `SUB` (il faut que son masque existe : `uv run python -m wmh_multisite.preproc.brainmask --subjects sub-XXX`).

In [ ]:
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd

from wmh_multisite.config import load_config
from wmh_multisite.preproc.brainmask import mask_paths, qc_row
from wmh_multisite.utils import io

SUB = "sub-000"
cfg = load_config()
P = mask_paths(cfg, SUB)

def canonical(path):
    img = nib.as_closest_canonical(io.load(path))
    return img.get_fdata(dtype=np.float32), img.header.get_zooms()[:3]

t1, z = canonical(P["t1w"])
brain, _ = canonical(P["brain_t1"])
head, _ = canonical(P["t1_head"])
VMAX = np.percentile(t1[t1 > 0], 99.5)
pd.Series(qc_row(cfg, SUB))

## 1. Lecture des chiffres

- `brain_ml` : volume du masque (cerveau + ventricules ; le liquide autour du cerveau est en partie exclu).
- `n_components`, `largest_component_frac` : un masque correct est **d'un seul tenant**.
- `brain_inside_head_frac` : part du cerveau contenue dans le masque de tête N4 (doit valoir ~1).
- `flair_coverage` : part du volume du cerveau qui reste dans le champ de vue de la FLAIR (qui ne couvre que ~144 mm de hauteur).

Dans les figures : **contour jaune** = bord du masque ; **voile jaune** = intérieur.

In [ ]:
def overlay(ax, img, mask, aspect, title=""):
    ax.imshow(img.T, origin="lower", cmap="gray", vmin=0, vmax=VMAX, aspect=aspect)
    ax.contourf(mask.T, levels=[0.5, 1.5], colors=["gold"], alpha=0.15)
    ax.contour(mask.T, levels=[0.5], colors=["gold"], linewidths=0.7)
    ax.set_title(title, fontsize=7); ax.axis("off")

def mosaic(axis, n, rows, title):
    extent = np.where(brain.any(axis=tuple(a for a in range(3) if a != axis)))[0]
    lo, hi = extent[0] - 4, extent[-1] + 4                      # a few slices beyond the mask on each side
    positions = np.linspace(max(lo, 0), min(hi, brain.shape[axis] - 1), n).astype(int)
    a, b = [d for d in range(3) if d != axis]
    fig, axes = plt.subplots(rows, int(np.ceil(n / rows)), figsize=(2.6 * n / rows, 2.8 * rows))
    for ax, k in zip(axes.ravel(), positions):
        sl = [slice(None)] * 3; sl[axis] = k
        overlay(ax, t1[tuple(sl)], brain[tuple(sl)], z[b] / z[a], f"{k * z[axis]:.0f} mm")
    for ax in axes.ravel()[len(positions):]:
        ax.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

## 2. Coupes axiales, du bas (gauche) vers le haut (droite)

In [ ]:
mosaic(axis=2, n=24, rows=4, title=f"{SUB} — coupes axiales (position depuis le bas du volume)")

## 3. Coupes coronales (de l'arrière vers l'avant) et sagittales (de gauche à droite)

In [ ]:
mosaic(axis=1, n=12, rows=2, title=f"{SUB} — coupes coronales")
mosaic(axis=0, n=12, rows=2, title=f"{SUB} — coupes sagittales")

## 4. Zooms sur les zones difficiles

Les erreurs typiques d'une extraction du cerveau se trouvent :
- à la **base du crâne** (tronc cérébral, cervelet, sinus) : où couper le tronc, ne pas inclure les sinus veineux ;
- aux **pôles temporaux et frontaux**, collés à l'os et aux orbites ;
- au **vertex** (sommet), où le cortex touche presque la voûte du crâne.

In [ ]:
ci = np.argwhere(brain)
center = ci.mean(axis=0).astype(int)
zmin, zmax = ci[:, 2].min(), ci[:, 2].max()
views = [
    ("base : tronc cérébral et cervelet (sagittale médiane)", brain[center[0], :, :], t1[center[0], :, :], z[2] / z[1],
     (slice(None), slice(max(zmin - 10, 0), zmin + 70))),
    ("pôles temporaux (axiale basse)", brain[:, :, zmin + 25], t1[:, :, zmin + 25], z[1] / z[0], (slice(None), slice(None))),
    ("lobes frontaux et orbites (axiale)", brain[:, :, zmin + 45], t1[:, :, zmin + 45], z[1] / z[0],
     (slice(None), slice(center[1], None))),
    ("vertex (coronale)", brain[:, center[1], :], t1[:, center[1], :], z[2] / z[0], (slice(None), slice(zmax - 50, zmax + 8))),
]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (title, m, img, asp, crop) in zip(axes, views):
    overlay(ax, img[crop], m[crop], asp, title)
plt.tight_layout(); plt.show()

## 5. Le masque sur la grille FLAIR

Masque transporté par la transformation rigide de l'étape 3.2 (interpolation `genericLabel`), superposé à la FLAIR brute. Les coupes FLAIR font 3 mm : le contour est en « marches » verticalement, c'est attendu.

In [ ]:
if P["brain_flair"].is_file():
    flair, zf = canonical(P["flair"])
    bf, _ = canonical(P["brain_flair"])
    VMAX_F = np.percentile(flair[flair > 0], 99.5)
    ks = np.linspace(0, flair.shape[2] - 1, 16).astype(int)
    fig, axes = plt.subplots(2, 8, figsize=(20, 5.6))
    for ax, k in zip(axes.ravel(), ks):
        ax.imshow(flair[:, :, k].T, origin="lower", cmap="gray", vmin=0, vmax=VMAX_F, aspect=zf[1] / zf[0])
        ax.contour(bf[:, :, k].T, levels=[0.5], colors=["gold"], linewidths=0.7)
        ax.set_title(f"coupe {k}", fontsize=7); ax.axis("off")
    fig.suptitle(f"{SUB} — FLAIR et masque du cerveau (grille FLAIR)"); plt.tight_layout(); plt.show()
else:
    print("Pas encore de masque sur la grille FLAIR (sujet non recalé, étape 3.5).")